# Phase 3: Model Prototyping (Professional Architectures)

**Objective**: Implement state-of-the-art architectures for bearing fault diagnosis.

### Models Included:
1. **Baseline MLP**: For statistical features.
2. **WDCNN (Wide Deep CNN)**: Industry standard for raw vibration signals. Uses a wide first kernel to suppress noise.
3. **LSTM (Long Short-Term Memory)**: Captures temporal dependencies in the vibration sequences.

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

PROCESSED_PATH = "../data/processed/cwru_features.parquet"
INTERIM_DIR = "../data/interim"
MODEL_SAVE_DIR = "../data/models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 1. Professional Architecture: WDCNN (Wide Deep CNN)
This architecture is highly effective because its wide first kernel (64 or 32) filters out high-frequency noise common in industrial environments.

In [ ]:
class WDCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=10):
        super(WDCNN, self).__init__()
        # Layer 1: Wide Kernel (64)
        self.layer1 = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=64, stride=16, padding=24),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        # Layer 2-5: Standard Kernels (3)
        self.layer2 = self._make_layer(16, 32)
        self.layer3 = self._make_layer(32, 64)
        self.layer4 = self._make_layer(64, 64)
        self.layer5 = self._make_layer(64, 64, last_pool=True)
        
        self.fc = nn.Sequential(
            nn.Linear(64, 100), # AdaptiveAvgPool makes this possible
            nn.ReLU(),
            nn.Linear(100, num_classes)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)

    def _make_layer(self, in_c, out_c, last_pool=False):
        layers = [
            nn.Conv1d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_c),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2) if not last_pool else nn.Identity()
        ]
        return nn.Sequential(*layers)

    def forward(self, x):
        # x shape: (batch, 1, 2048)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

wdcnn = WDCNN(num_classes=10).to(DEVICE)
print(wdcnn)

## 2. Professional Architecture: LSTM
Recurrent networks can capture the temporal patterns in the vibration time series.

In [ ]:
class FaultLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, num_classes=10):
        super(FaultLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        # x: (batch, seq_len, 1)
        out, _ = self.lstm(x)
        # Use the last hidden state
        out = self.fc(out[:, -1, :])
        return out

lstm_model = FaultLSTM().to(DEVICE)
print(lstm_model)

## 3. Discussion on Complexity
- **MLP**: Simple, fast, but sensitive to feature engineering quality.
- **WDCNN**: The "Gold Standard" for CWRU. Its hierarchical structure and wide kernel provide excellent noise immunity.
- **LSTM**: Deep temporal understanding, but can be slower to train on very long sequences.

Running **WDCNN** on the RTX 5070 will be extremely fast and should yield accuracy > 99%.